# ROS 2 / ACT：選択設定を固定した再実行

初回は25/50→50/50成功しましたが、保存先ダイアログの待機中に一時環境が終了し、重みと生の軌跡を失いました。このノートは固定設定で再学習・独立評価し、成果物を再作成します。初回の集計と新しい結果は別々に保存します。提出済みのMuJoCo搬送作品とは別です。

「ランタイム → ランタイムのタイプを変更」でGPUを選択し、上から順に実行してください。実行環境の初回準備には時間がかかります。既存契約の範囲で利用し、追加購入を自動で行う処理はありません。

このノートブックはソースから生成した**再現用**で、結果を装った事前出力は含みません。実測結果は別添のevidence ZIPに保存しています。

In [ ]:
import sys,subprocess
from pathlib import Path
import torch
assert Path("/content").exists(), "Google Colabで実行してください"
assert torch.cuda.is_available(), "GPUランタイムへ接続してください"
print(sys.version,torch.__version__,torch.cuda.get_device_name(0))

## 1. ROS 2 Humble用の環境をColab内に準備
Colab本体の新しいUbuntuと、教材のUbuntu 22.04を分離します。署名検証は有効のまま使います。

In [ ]:
"""Prepare isolated Ubuntu 22.04 ROS Humble userspace on a Colab VM.

No local Docker, no account tokens, no external logging or publication.
Run only on the disposable Colab VM; downloads are from Ubuntu/ROS upstream.
"""
from pathlib import Path
import hashlib
import json
import os
import subprocess
import urllib.request

BASE = Path('/content/pai_ros2')
ROOT = BASE/'jammy'
BASE.mkdir(exist_ok=True)


def run(args, name, env=None):
    print(name, flush=True)
    logfile = BASE/f'{name}.log'
    with logfile.open('w') as f:
        result = subprocess.run(args, stdout=f, stderr=subprocess.STDOUT, env=env)
    if result.returncode:
        print(logfile.read_text(errors='replace')[-10000:], flush=True)
        raise RuntimeError(f'{name}: exit {result.returncode}')
    print(f'{name}: OK', flush=True)


run(['apt-get', 'update', '-qq'], 'host_apt_update')
run(['apt-get', 'install', '-y', '-qq', 'git', 'xz-utils', 'libtalloc-dev', 'build-essential', 'pkg-config'], 'host_proot_dependencies')
if not (BASE/'proot-src/.git').exists():
    run(['git','clone','--depth','1','--branch','v5.5.0','https://github.com/proot-me/proot.git',str(BASE/'proot-src')], 'proot_source')
run(['make','-C',str(BASE/'proot-src/src'),'-j2','proot'], 'proot_build')
name = 'ubuntu-22.04-minimal-cloudimg-amd64-root.tar.xz'
url = 'https://cloud-images.ubuntu.com/minimal/releases/jammy/release/'
if not (ROOT/'bin/bash').exists():
    print('Downloading official Ubuntu 22.04 rootfs', flush=True)
    sums = urllib.request.urlopen(url+'SHA256SUMS').read().decode()
    expected = next(line.split()[0] for line in sums.splitlines() if line.split()[-1].lstrip('*') == name)
    archive = BASE/name
    urllib.request.urlretrieve(url+name, archive)
    actual = hashlib.sha256(archive.read_bytes()).hexdigest()
    assert actual == expected, 'Ubuntu rootfs checksum mismatch'
    ROOT.mkdir(exist_ok=True)
    run(['tar', '-xJf', str(archive), '-C', str(ROOT), '--exclude=dev/*'], 'unpack_jammy')
    (BASE/'rootfs_manifest.json').write_text(json.dumps({'url':url+name,'sha256':actual},indent=2))
resolver = ROOT/'etc/resolv.conf'
if resolver.is_symlink():
    resolver.unlink()
resolver.write_text(Path('/etc/resolv.conf').read_text())
(ROOT/'usr/sbin/policy-rc.d').write_text('#!/bin/sh\nexit 101\n')
(ROOT/'usr/sbin/policy-rc.d').chmod(0o755)
(ROOT/'etc/apt/sources.list').write_text('deb https://archive.ubuntu.com/ubuntu jammy main universe restricted multiverse\ndeb https://archive.ubuntu.com/ubuntu jammy-updates main universe restricted multiverse\ndeb https://security.ubuntu.com/ubuntu jammy-security main universe restricted multiverse\n')
prefix = [str(BASE/'proot-src/src/proot'),'-0','-r',str(ROOT),'-b','/dev','-b','/proc','-b','/sys','-b',f'{BASE}:/work','-w','/root','/usr/bin/env','-i','HOME=/root','PATH=/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin','DEBIAN_FRONTEND=noninteractive','LANG=C.UTF-8','LC_ALL=C.UTF-8']
(BASE/'proot_prefix.json').write_text(json.dumps(prefix))
run(prefix+['bash','-c','apt-get update -qq && apt-get install -y -qq ca-certificates curl gnupg git python3-pip python3-venv build-essential cmake xvfb'], 'jammy_tools')
# Official apt transport with an HTTPS-fetched key. Signature verification stays enabled.
run(prefix+['bash','-c','curl -fsSL https://raw.githubusercontent.com/ros/rosdistro/master/ros.key -o /usr/share/keyrings/ros-archive-keyring.gpg && echo "deb [arch=amd64 signed-by=/usr/share/keyrings/ros-archive-keyring.gpg] http://packages.ros.org/ros2/ubuntu jammy main" > /etc/apt/sources.list.d/ros2.list && apt-get update --error-on=any -qq'], 'ros_repository')
run(prefix+['apt-get','install','-y','-qq','ros-humble-ros-base','ros-humble-ros-gz','ros-humble-ign-ros2-control','ros-humble-ros2-controllers','ros-humble-xacro','ros-humble-robot-state-publisher','ros-humble-dynamixel-sdk','python3-colcon-common-extensions'], 'ros_gazebo')
run(prefix+['bash','-c','source /opt/ros/humble/setup.bash && python3 -c "import rclpy; print(rclpy.__file__)" && ign gazebo --versions'], 'smoke_ros')
print('ROS_BOOTSTRAP_COMPLETE',flush=True)


## 2. 教材の固定バージョンを取得してロボットを構築
ロボットモデル、コントローラー、Gazebo世界は教材のものです。GUIのみ省いて起動します。

In [ ]:
"""Run after bootstrap_colab.py, in the same Colab kernel."""
from pathlib import Path
import subprocess
import json
import sys

BASE = Path('/content/pai_ros2')
prefix = json.loads((BASE/'proot_prefix.json').read_text())

def run(args, name):
    print(name, flush=True)
    with (BASE/f'{name}.log').open('w') as log:
        p = subprocess.run(args, stdout=log, stderr=subprocess.STDOUT)
    if p.returncode:
        print((BASE/f'{name}.log').read_text()[-8000:], flush=True)
        raise RuntimeError(name)
    print(name+' OK', flush=True)

repos = [('https://github.com/AI-Robot-Book-Humble/crane_plus.git',
          BASE/'ws/src/crane_plus', '19c1d6069126cd27488bf1bbefd410704bfc515d'),
         ('https://github.com/matsuolab/pai_training.git',
          BASE/'pai_training','b4d6b5459066a80433475a83567504988875248a')]
for url, path, sha in repos:
    if not (path/'.git').exists():
        run(['git','clone',url,str(path)],'clone_'+path.name)
    run(['git','-C',str(path),'checkout',sha],'checkout_'+path.name)
run(['git','-C',str(BASE/'pai_training'),'submodule','update','--init','lerobot'],'lerobot_source')
# Keep the course robot, controllers and world intact; only suppress the GUI.
launch = BASE/'ws/src/crane_plus/crane_plus_gazebo/launch/no_moveit_crane_plus_with_table.launch.py'
text = launch.read_text().replace("'ign gazebo -r'", "'ign gazebo -r -s'")
launch.with_name('headless_crane_plus.launch.py').write_text(text)
run(prefix+['bash','-c','source /opt/ros/humble/setup.bash && cd /work/ws && colcon build --packages-select crane_plus_description crane_plus_control crane_plus_moveit_config crane_plus_gazebo --executor sequential --cmake-args -DBUILD_TESTING=OFF'],'build_crane')
with (BASE/'robot.urdf').open('w') as output:
    subprocess.run(prefix+['bash','-c','source /opt/ros/humble/setup.bash && source /work/ws/install/setup.bash && xacro /work/ws/install/crane_plus_description/share/crane_plus_description/urdf/crane_plus.urdf.xacro use_gazebo:=true use_ignition:=true gz_control_config_package:=crane_plus_control gz_control_config_file_path:=config/crane_plus_controllers.yaml'],stdout=output,check=True)
# Training uses Colab's native CUDA stack; ROS remains isolated in Ubuntu 22.04.
run([sys.executable,'-m','pip','install','-q','draccus==0.10.0','jsonlines','datasets<4','einops','safetensors','huggingface-hub<1','deepdiff','termcolor'],'act_dependencies')
sys.path.insert(0,str(BASE/'pai_training/lerobot'))
import torch
from lerobot.common.policies.act.modeling_act import ACTPolicy
from lerobot.common.policies.act.configuration_act import ACTConfig
print('ACT_IMPORT_OK',torch.__version__,torch.cuda.get_device_name(0),flush=True)
(BASE/'environment.json').write_text(json.dumps({'torch':torch.__version__,'python':sys.version,'gpu':torch.cuda.get_device_name(0)},indent=2))
print('EXPERIMENT_SETUP_COMPLETE',flush=True)


## 3. 独自実装を保存
各セルは表示されたPythonファイルを保存します。コードの内容を確認できるよう、圧縮せず記載しています。

In [ ]:
SRC=Path("/content/pai_ros2/src"); SRC.mkdir(exist_ok=True)

In [ ]:
%%writefile /content/pai_ros2/src/data_alignment.py
"""Explicit joint-name and causal time alignment for the lesson's ROS data."""
import numpy as np

JOINTS = ('crane_plus_joint1', 'crane_plus_joint2', 'crane_plus_joint3',
          'crane_plus_joint4', 'crane_plus_joint_hand')


def reorder(names, positions, required=JOINTS):
    if len(names) != len(positions) or len(names) != len(set(names)):
        raise ValueError('Invalid or duplicate joint names')
    values = dict(zip(names, positions))
    if any(name not in values for name in required):
        raise ValueError('Missing joint: refusing silent zero fill')
    result = np.asarray([values[name] for name in required], dtype=np.float32)
    if not np.isfinite(result).all():
        raise ValueError('Non-finite joint state')
    return result


def causal_sample(times_ns, values, target_ns, max_age_ns=None):
    times = np.asarray(times_ns, dtype=np.int64)
    target = np.asarray(target_ns, dtype=np.int64)
    values = np.asarray(values)
    if not len(times) or len(times) != len(values) or np.any(np.diff(times) < 0):
        raise ValueError('Empty, mismatched or unsorted series')
    idx = np.searchsorted(times, target, side='right')-1
    if np.any(idx < 0):
        raise ValueError('No past observation; future leakage forbidden')
    age = target-times[idx]
    if max_age_ns is not None and np.any(age > max_age_ns):
        raise ValueError('Stale observation')
    return values[idx], age


def uniform_times(start_ns, end_ns, fps=20):
    if fps <= 0:
        raise ValueError('Invalid sampling frequency')
    period = 1_000_000_000//fps
    if fps <= 0 or period*fps != 1_000_000_000 or end_ns <= start_ns:
        raise ValueError('Invalid sampling grid')
    return np.arange(start_ns, end_ns, period, dtype=np.int64)


def action_offsets(chunk_size=16, fps=20):
    """Index zero corresponds to the command issued at observation time."""
    return [i/fps for i in range(chunk_size)]


def action_chunk(actions, index, chunk_size=16):
    """End padding is explicitly masked, not added as real episode frames."""
    actions = np.asarray(actions)
    if not 0 <= index < len(actions):
        raise IndexError(index)
    indices = np.arange(index, index+chunk_size)
    return actions[np.minimum(indices, len(actions)-1)], indices >= len(actions)


In [ ]:
%%writefile /content/pai_ros2/src/test_data_alignment.py
import unittest
import numpy as np
from data_alignment import JOINTS, reorder, causal_sample, uniform_times, action_offsets, action_chunk


class AlignmentTests(unittest.TestCase):
    def test_shuffled_names(self):
        order = [4, 2, 0, 3, 1]
        np.testing.assert_equal(reorder([JOINTS[i] for i in order], order), np.arange(5))

    def test_missing_joint_is_not_zero(self):
        with self.assertRaises(ValueError):
            reorder(JOINTS[:-1], [0]*4)

    def test_no_future_and_stale_data(self):
        with self.assertRaises(ValueError):
            causal_sample([100, 200], [1, 2], [99])
        with self.assertRaises(ValueError):
            causal_sample([100, 200], [1, 2], [300], max_age_ns=50)
        np.testing.assert_equal(causal_sample([100, 200], [1, 2], [199,200])[0], [1,2])

    def test_fixed_period(self):
        t = uniform_times(1_700_000_000_000_000_000, 1_700_000_001_030_000_000)
        np.testing.assert_equal(np.diff(t), 50_000_000)

    def test_invalid_frequency(self):
        for fps in [0,-1,3]:
            with self.assertRaises(ValueError): uniform_times(0,1_000_000_000,fps)

    def test_action_zero_is_current(self):
        self.assertEqual(action_offsets()[0], 0)
        np.testing.assert_allclose(np.diff(action_offsets()), .05)

    def test_end_mask(self):
        data, mask = action_chunk(np.arange(5)[:,None],3,4)
        np.testing.assert_equal(data[:,0], [3,4,4,4])
        np.testing.assert_equal(mask, [False,False,True,True])


if __name__ == '__main__':
    unittest.main()


In [ ]:
%%writefile /content/pai_ros2/src/act_experiment.py
"""Pinned upstream ACT, measured ROS data, episode-level separation, CUDA.

Data loader is a transparent in-memory adapter (no image/video data). The lesson
baseline reproduces its model, optimizer, steps, batching, joint-order and target
offset conventions, rather than claiming byte-identical ROS bag conversion.
"""
import argparse
import json
import random
import sys
import time
import hashlib
from pathlib import Path
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

BASE=Path('/content/pai_ros2')
sys.path.insert(0,str(BASE/'pai_training/lerobot'))
from lerobot.common.policies.act.configuration_act import ACTConfig
from lerobot.common.policies.act.modeling_act import ACTPolicy
from lerobot.configs.types import FeatureType,PolicyFeature
from data_alignment import causal_sample,uniform_times

PERM=[0,1,3,2,4]  # Upstream teleoperation and infer.py arm joint ordering.
TRIALS={
 'lesson_baseline':{'steps':200,'batch':2,'episodes':20,'baseline':True},
 'order_only_200':{'steps':200,'batch':2,'episodes':20,'baseline':False,'legacy_time':True,'legacy_padding':True},
 'time_only_200':{'steps':200,'batch':2,'episodes':20,'baseline':True,'legacy_time':False,'legacy_padding':True},
 'aligned_200':{'steps':200,'batch':2,'episodes':20,'baseline':False},
 'aligned_2000':{'steps':2000,'batch':16,'episodes':20,'baseline':False},
 'diverse_2000':{'steps':2000,'batch':16,'episodes':80,'baseline':False},
}

class RecordedDataset(Dataset):
    def __init__(self, directory, cfg):
        self.baseline=cfg['baseline']; self.episodes=[]; self.index=[]
        legacy_time=cfg.get('legacy_time',self.baseline)
        legacy_padding=cfg.get('legacy_padding',self.baseline)
        for seed in range(cfg['episodes']):
            obj=json.loads((Path(directory)/f'episode_{seed:05d}.json').read_text())
            if not(obj['teacher'] and obj['success'] and obj['valid']):
                raise ValueError(f'Quality-gate rejected episode {seed}')
            trace=obj['trace']
            times=np.array([r['t'] for r in trace])
            if np.max(np.diff(times))>.075:
                raise ValueError(f'Irregular recording cadence: seed {seed}')
            states=np.array([r['raw'] if self.baseline else r['q'] for r in trace],dtype=np.float32)
            actions=np.array([r['action'] for r in trace],dtype=np.float32)
            if states.shape[1]!=5: raise ValueError('Unexpected JointState shape')
            if self.baseline: actions=actions[:,PERM]
            times_ns=np.rint(times*1e9).astype(np.int64)
            grid=(np.linspace(times_ns[0],times_ns[-1],int((times[-1]-times[0])*20)).astype(np.int64)
                  if legacy_time else uniform_times(times_ns[0],times_ns[-1],20))
            states,_=causal_sample(times_ns,states,grid,max_age_ns=75_000_000)
            actions,_=causal_sample(times_ns,actions,grid,max_age_ns=75_000_000)
            self.episodes.append([states,actions])
        if legacy_padding:
            maxlen=max(len(s) for s,a in self.episodes)
            for pair in self.episodes:
                for j in range(2):
                    pair[j]=np.pad(pair[j],((0,maxlen-len(pair[j])),(0,0)),mode='edge')
        for ep,(states,actions) in enumerate(self.episodes):
            self.index.extend((ep,i) for i in range(len(states)))
        self.offsets=np.arange(-2,30,2) if legacy_time else np.arange(16)
        self.stats={}
        for key, col in [('observation.state',0),('observation.environment_state',0),('action',1)]:
            arr=np.concatenate([pair[col] for pair in self.episodes])
            self.stats[key]={'mean':arr.mean(0),'std':arr.std(0),'min':arr.min(0),'max':arr.max(0)}

    def __len__(self): return len(self.index)

    def __getitem__(self,index):
        ep,i=self.index[index]; states,actions=self.episodes[ep]
        indices=i+self.offsets
        # LeRobot masks out-of-episode action queries.
        pad=(indices<0)|(indices>=len(actions))
        a=actions[np.clip(indices,0,len(actions)-1)]
        s=torch.from_numpy(states[i].copy())
        return {'observation.state':s,'observation.environment_state':s.clone(),
                'action':torch.from_numpy(a.copy()),'action_is_pad':torch.from_numpy(pad)}

def make_policy(stats=None, n_action_steps=1, ensemble=None):
    cfg=ACTConfig(input_features={
        'observation.state':PolicyFeature(FeatureType.STATE,(5,)),
        'observation.environment_state':PolicyFeature(FeatureType.ENV,(5,))},
        output_features={'action':PolicyFeature(FeatureType.ACTION,(5,))},
        chunk_size=16,n_action_steps=n_action_steps,temporal_ensemble_coeff=ensemble,device='cuda')
    return ACTPolicy(cfg,dataset_stats=stats).cuda()

def train(name,directory,seed=2026):
    cfg=TRIALS[name]
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    torch.set_num_threads(2)
    torch.backends.cudnn.benchmark=False
    dataset=RecordedDataset(directory,cfg)
    policy=make_policy(dataset.stats); policy.train()
    optimizer=torch.optim.Adam(policy.parameters(),lr=1e-4)
    loader=DataLoader(dataset,batch_size=cfg['batch'],shuffle=True,num_workers=0,drop_last=True)
    out=BASE/'results'/name; out.mkdir(parents=True,exist_ok=True)
    metadata={**cfg,'seed':seed,'lr':1e-4,'samples':len(dataset),'torch':torch.__version__,
              'raw_joint_names':json.loads((Path(directory)/'episode_00000.json').read_text())['trace'][0]['raw_names'],
              'normalization':'train episodes only, population mean/std, no variance floor',
              'loader_deviation':'in-memory 20Hz ROS records; no rosbag/LeRobot disk conversion'}
    metadata['training_source_sha256']=hashlib.sha256(Path(__file__).read_bytes()).hexdigest()
    metadata['data_sha256']={f'episode_{s:05d}.json':hashlib.sha256((Path(directory)/f'episode_{s:05d}.json').read_bytes()).hexdigest() for s in range(cfg['episodes'])}
    (out/'trial.json').write_text(json.dumps(metadata,indent=2))
    tick=time.monotonic(); step=0
    with (out/'loss.jsonl').open('w') as log:
        while step<cfg['steps']:
            for batch in loader:
                batch={k:v.cuda() for k,v in batch.items()}
                loss, info=policy.forward(batch)
                if not torch.isfinite(loss): raise RuntimeError(f'Non-finite loss at {step}')
                optimizer.zero_grad(); loss.backward(); optimizer.step(); step+=1
                row={'step':step,'loss':float(loss.detach()),'seconds':time.monotonic()-tick}
                log.write(json.dumps(row)+'\n')
                if step%100==0: print(name,json.dumps(row),flush=True); log.flush()
                if step>=cfg['steps']: break
    policy.eval(); policy.save_pretrained(out/'checkpoint')
    metadata.update({'training_seconds':time.monotonic()-tick,'final_loss':row['loss'],
                     'parameters':sum(p.numel() for p in policy.parameters()),'complete':True})
    (out/'trial.json').write_text(json.dumps(metadata,indent=2))
    print('TRAIN_COMPLETE',name,flush=True)

def main():
    p=argparse.ArgumentParser(); p.add_argument('--trial',choices=list(TRIALS)+['all'],default='all')
    p.add_argument('--data',default=str(BASE/'data/train')); args=p.parse_args()
    for name in (TRIALS if args.trial=='all' else [args.trial]):
        finished=BASE/'results'/name/'trial.json'
        if finished.exists() and json.loads(finished.read_text()).get('complete'):
            meta=json.loads(finished.read_text())
            if meta['training_source_sha256']!=hashlib.sha256(Path(__file__).read_bytes()).hexdigest():
                raise RuntimeError('Training source changed: use a new trial name')
            for filename,digest in meta['data_sha256'].items():
                if hashlib.sha256((Path(args.data)/filename).read_bytes()).hexdigest()!=digest:
                    raise RuntimeError('Training data changed: use a new trial name')
            continue
        train(name,args.data)

if __name__=='__main__': main()


In [ ]:
%%writefile /content/pai_ros2/src/policy_service.py
"""Loopback-only bridge between native Colab CUDA and isolated ROS Python."""
import json
import traceback
from http.server import BaseHTTPRequestHandler,HTTPServer
import torch
from act_experiment import BASE,PERM,make_policy
from safetensors.torch import load_model

torch.set_num_threads(2)
policy=None; baseline=False

class Handler(BaseHTTPRequestHandler):
    def log_message(self,*args): pass

    def do_POST(self):
        global policy,baseline
        try:
            data=json.loads(self.rfile.read(int(self.headers['Content-Length'])))
            if self.path=='/load':
                trial=data['trial']
                path=(BASE/'results'/trial).resolve()
                if path.parent != (BASE/'results').resolve(): raise ValueError('Invalid trial')
                meta=json.loads((path/'trial.json').read_text())
                policy=make_policy(n_action_steps=data.get('n_action_steps',1),ensemble=data.get('ensemble'))
                load_model(policy,str(path/'checkpoint/model.safetensors'),strict=True)
                policy.eval(); baseline=meta['baseline']; policy.reset(); result={'loaded':trial}
            elif self.path=='/reset':
                policy.reset(); result={'reset':True}
            elif self.path=='/predict':
                x=torch.tensor(data['state'],dtype=torch.float32,device='cuda')
                if baseline: x=x[PERM]  # Reproduce upstream infer.py's name-based input order.
                batch={'observation.state':x[None],'observation.environment_state':x[None]}
                with torch.inference_mode(): action=policy.select_action(batch)[0]
                if baseline: action=action[PERM]  # Canonical physical publisher order.
                result={'action':action.cpu().tolist()}
            else: raise ValueError('Unknown endpoint')
            body=json.dumps(result).encode(); self.send_response(200)
        except Exception:
            body=json.dumps({'error':traceback.format_exc()}).encode(); self.send_response(500)
        self.send_header('Content-Type','application/json'); self.send_header('Content-Length',str(len(body)))
        self.end_headers(); self.wfile.write(body)

if __name__=='__main__':
    print('POLICY_SERVICE_READY',flush=True)
    HTTPServer(('127.0.0.1',8765),Handler).serve_forever()


In [ ]:
%%writefile /content/pai_ros2/src/ros_trials.py
"""Actual Gazebo/ROS feedback collection and closed-loop evaluation (Python 3.10).

Teacher reproduces keyboard increments; it is not a human demonstration.
Results are simulator measurements, not real-robot performance.
"""
import argparse
import json
import time
import traceback
import urllib.request
from pathlib import Path
import numpy as np
import rclpy
from rclpy.node import Node
from rclpy.parameter import Parameter
from sensor_msgs.msg import JointState
from trajectory_msgs.msg import JointTrajectory, JointTrajectoryPoint
from data_alignment import JOINTS, reorder

TARGET = np.array([0., 0., .6, 0., -.5236])
LOW = np.deg2rad(np.array([0,45.45,3.52,44.57,109.38])-150)
HIGH = np.deg2rad(np.array([300,252.20,290.62,251.32,188.27])-150)

def rpc(path, payload):
    req=urllib.request.Request('http://127.0.0.1:8765/'+path,
        data=json.dumps(payload).encode(),headers={'Content-Type':'application/json'})
    with urllib.request.urlopen(req, timeout=30) as f:
        return json.load(f)

class Robot(Node):
    def __init__(self):
        super().__init__('pai_measured_trials')
        self.set_parameters([Parameter('use_sim_time', value=True)])
        self.q=None; self.raw=None; self.names=None; self.stamp=None
        self.create_subscription(JointState,'/joint_states',self.receive,10)
        self.arm=self.create_publisher(JointTrajectory,'/crane_plus_arm_controller/joint_trajectory',10)
        self.hand=self.create_publisher(JointTrajectory,'/crane_plus_gripper_controller/joint_trajectory',10)

    def receive(self,msg):
        self.q=reorder(msg.name,msg.position).astype(float)
        self.raw=list(msg.position); self.names=list(msg.name)
        self.stamp=msg.header.stamp.sec+msg.header.stamp.nanosec*1e-9

    def now(self):
        return self.get_clock().now().nanoseconds/1e9

    def wait(self, until, wall_timeout=45):
        deadline=time.monotonic()+wall_timeout
        while self.now()<until or self.q is None:
            if time.monotonic()>deadline:
                raise RuntimeError('Simulation clock or JointState timeout')
            rclpy.spin_once(self,timeout_sec=.01)
        # Consume queued feedback; reject a stalled feedback stream.
        for _ in range(3): rclpy.spin_once(self,timeout_sec=0.)
        if self.stamp is None or self.now()-self.stamp>.20:
            raise RuntimeError('JointState older than 200 ms')

    def command(self,q):
        if not np.isfinite(q).all(): raise ValueError('Non-finite action')
        for pub,names,values in [(self.arm,JOINTS[:4],q[:4]),(self.hand,JOINTS[4:],q[4:])]:
            msg=JointTrajectory(); msg.joint_names=list(names)
            point=JointTrajectoryPoint(); point.positions=[float(v) for v in values]
            point.time_from_start.nanosec=100_000_000
            msg.points=[point]; pub.publish(msg)

    def reset_to(self,q):
        start=self.now(); prev=self.q.copy(); stable_since=None
        for k in range(100):
            self.command(q); self.wait(start+(k+1)*.05)
            stationary=np.max(np.abs(self.q-prev))/.05<.05
            if np.max(np.abs(self.q-q))<.005 and stationary:
                if stable_since is None: stable_since=self.now()
                if self.now()-stable_since>=.3: return
            else: stable_since=None
            prev=self.q.copy()
        raise RuntimeError('Reset did not stably reach requested initial posture')

def episode(robot, seed, teacher=False, oracle=False):
    rng=np.random.default_rng(seed)
    # The first twenty episodes reproduce the lesson's zero-start data.
    q0=np.r_[np.zeros(4) if teacher and seed<20 else rng.uniform(-.08,.08,4),-.5236]
    robot.reset_to(q0)
    if not teacher and not oracle: rpc('reset',{})
    initial=robot.q.copy(); start=robot.now(); hold_start=None
    trace=[]; limits=0; completion=None; prev_q=robot.q.copy(); prev_t=start
    interval=float(rng.uniform(.10,.17)) if teacher else None
    for k in range(160):
        scheduled=start+k*.05
        robot.wait(scheduled)
        t=robot.now(); q=robot.q.copy(); raw=robot.raw.copy()
        if t-start>8.0:
            break
        velocity=(q-prev_q)/max(t-prev_t,1e-6) if k else np.zeros(5)
        if np.max(np.abs(q-TARGET))<=.05 and np.max(np.abs(velocity))<=.10:
            if hold_start is None: hold_start=t
            if t-hold_start>=.5 and completion is None: completion=t-start
        else: hold_start=None
        latency=0.
        if oracle:
            action=TARGET.copy()
        elif teacher:
            count=min(12,int((t-start)/interval)+1)
            fraction=count/12
            # 0.05-rad target increments, plus recovery from varied initial posture.
            action=q0+(TARGET-q0)*fraction
        else:
            tic=time.perf_counter(); action=np.asarray(rpc('predict',{'state':q.tolist()})['action'])
            latency=(time.perf_counter()-tic)*1000
        violation=bool(np.any(action<LOW)|np.any(action>HIGH))
        limits+=int(violation)
        # Hard joint limits shared by every method; all interventions counted.
        applied=np.clip(action,LOW,HIGH)
        robot.command(applied)
        trace.append({'t':t-start,'q':q.tolist(),'raw':raw,'raw_names':robot.names,
                      'action':action.tolist(),'applied':applied.tolist(),'latency_ms':latency,
                      'feedback_age_s':t-robot.stamp,'schedule_delay_s':t-scheduled})
        prev_q=q; prev_t=t
        # Keep 0.5 s of stable terminal data in demonstrations, already required by success.
        if completion is not None: break
    qs=np.array([r['q'] for r in trace]); ts=np.array([r['t'] for r in trace])
    valid=bool(np.isfinite(qs).all() and np.all(np.diff(ts)>0))
    jerk=np.gradient(np.gradient(np.gradient(qs,ts,axis=0),ts,axis=0),ts,axis=0)
    result={'seed':seed,'teacher':teacher,'oracle':oracle,'success':completion is not None,'valid':valid,
            'initial':initial.tolist(),'terminal_rmse_rad':float(np.sqrt(np.mean((qs[-1]-TARGET)**2))),
            'completion_sim_seconds':completion,'joint_jerk_rms':float(np.sqrt(np.mean(jerk[3:-3]**2))),
            'command_limit_violations':limits,'inference_latency_ms_mean':float(np.mean([r['latency_ms'] for r in trace])),
            'max_schedule_delay_s':max(r['schedule_delay_s'] for r in trace),'trace':trace}
    return result

def main():
    p=argparse.ArgumentParser(); p.add_argument('--out',required=True)
    p.add_argument('--teacher',action='store_true'); p.add_argument('--start',type=int,default=0)
    p.add_argument('--oracle',action='store_true')
    p.add_argument('--count',type=int,default=20); args=p.parse_args()
    out=Path(args.out); out.mkdir(parents=True,exist_ok=True)
    rclpy.init(); robot=Robot(); robot.wait(1.,wall_timeout=120)
    print('JOINT_STATE_ORDER',robot.names,flush=True)
    for seed in range(args.start,args.start+args.count):
        target=out/f'episode_{seed:05d}.json'
        if target.exists(): continue
        tic=time.monotonic()
        try:
            result=episode(robot,seed,args.teacher,args.oracle)
        except Exception:
            result={'seed':seed,'teacher':args.teacher,'success':False,'valid':False,
                    'error':traceback.format_exc(),'trace':[]}
        result['wall_seconds']=time.monotonic()-tic
        temp=target.with_suffix('.tmp'); temp.write_text(json.dumps(result)); temp.replace(target)
        print(json.dumps({k:v for k,v in result.items() if k!='trace'}),flush=True)
        if args.teacher and not result['valid']:
            raise RuntimeError('Teacher collection failed; no training on rejected data')
    robot.destroy_node(); rclpy.shutdown()

if __name__=='__main__': main()


In [ ]:
%%writefile /content/pai_ros2/src/start_colab_runtime.py
"""Start the course Gazebo world, then collect one non-evaluation smoke episode."""
from pathlib import Path
import subprocess
import json
import os

BASE=Path('/content/pai_ros2')
prefix=json.loads((BASE/'proot_prefix.json').read_text())
setup='source /opt/ros/humble/setup.bash && source /work/ws/install/setup.bash && '
assert (BASE/'ws/install/setup.bash').exists(), 'Build must succeed first'
assert (BASE/'ws/install/crane_plus_gazebo/share/crane_plus_gazebo/launch/headless_crane_plus.launch.py').exists(), 'Gazebo package must build successfully'
log=(BASE/'gazebo.log').open('w')
p=subprocess.Popen(prefix+['bash','-c',setup+'ros2 launch crane_plus_gazebo headless_crane_plus.launch.py'],
                   stdout=log,stderr=subprocess.STDOUT,start_new_session=True)
(BASE/'gazebo_process.json').write_text(json.dumps({'pid':p.pid,'command':'headless_crane_plus.launch.py'}))
print('GAZEBO_PID',p.pid,flush=True)
with (BASE/'probe.log').open('w') as f:
    probe=subprocess.run(prefix+['bash','-c',setup+'python3 /work/src/ros_trials.py --teacher --start 9999 --count 1 --out /work/data/probe'],stdout=f,stderr=subprocess.STDOUT)
print((BASE/'probe.log').read_text()[-5000:],flush=True)
if probe.returncode:
    print((BASE/'gazebo.log').read_text()[-8000:],flush=True)
    raise RuntimeError('Gazebo/ROS probe failed; no training started')
result=json.loads((BASE/'data/probe/episode_09999.json').read_text())
assert result['success'] and result['valid'], 'Teacher must solve the task first'
print('GAZEBO_ROLLOUT_SMOKE_OK',flush=True)


In [ ]:
%%writefile /content/pai_ros2/src/smoke_gpu.py
"""Integration smoke test only; synthetic data are never performance evidence."""
import json
import time
import numpy as np
import torch
from act_experiment import make_policy,BASE

torch.manual_seed(2026); torch.set_num_threads(2)
stats={k:{'mean':np.zeros(5,dtype=np.float32),'std':np.ones(5,dtype=np.float32)}
       for k in ['observation.state','observation.environment_state','action']}
start=time.monotonic(); p=make_policy(stats)
batch={'observation.state':torch.zeros((2,5),device='cuda'),
       'observation.environment_state':torch.zeros((2,5),device='cuda'),
       'action':torch.zeros((2,16,5),device='cuda'),
       'action_is_pad':torch.zeros((2,16),dtype=torch.bool,device='cuda')}
p.train(); loss,_=p(batch); assert torch.isfinite(loss); loss.backward()
p.eval(); p.reset()
with torch.inference_mode(): a=p.select_action(batch)
assert a.shape==(2,5) and torch.isfinite(a).all()
result={'status':'passed','purpose':'synthetic integration smoke test, NOT accuracy evaluation',
        'device':torch.cuda.get_device_name(0),'loss':float(loss.detach()),
        'elapsed_seconds':time.monotonic()-start}
(BASE/'gpu_smoke.json').write_text(json.dumps(result,indent=2))
print('GPU_SMOKE_OK',json.dumps(result),flush=True)


In [ ]:
%%writefile /content/pai_ros2/src/run_comparison.py
"""Finite preregistered comparison; chooses on validation before accessing test."""
import json
import subprocess
import sys
import time
import urllib.request
from pathlib import Path
from statistics import mean
import hashlib

BASE=Path('/content/pai_ros2'); SRC=BASE/'src'
prefix=json.loads((BASE/'proot_prefix.json').read_text())

def call(args,name):
    print('START',name,flush=True)
    with (BASE/f'{name}.log').open('w') as f:
        p=subprocess.run(args,stdout=f,stderr=subprocess.STDOUT)
    if p.returncode:
        print((BASE/f'{name}.log').read_text()[-4000:],flush=True)
        raise RuntimeError(name)
    print('DONE',name,flush=True)

def rpc(path,data):
    r=urllib.request.Request('http://127.0.0.1:8765/'+path,data=json.dumps(data).encode(),
                             headers={'Content-Type':'application/json'})
    with urllib.request.urlopen(r,timeout=120) as f: return json.load(f)

def evaluate(name,cfg,start,count):
    out=BASE/'evaluation'/name; out.mkdir(parents=True,exist_ok=True)
    signature={'config':cfg,'start_seed':start,'count':count,
               'runner_sha256':hashlib.sha256((SRC/'ros_trials.py').read_bytes()).hexdigest(),
               'policy_service_sha256':hashlib.sha256((SRC/'policy_service.py').read_bytes()).hexdigest()}
    if not cfg.get('oracle'):
        signature['checkpoint_sha256']=hashlib.sha256((BASE/'results'/cfg['trial']/'checkpoint/model.safetensors').read_bytes()).hexdigest()
    config_file=out/'config.json'
    if config_file.exists() and json.loads(config_file.read_text())!=signature:
        raise RuntimeError('Refusing to mix evaluation results from different code or checkpoints: '+name)
    config_file.write_text(json.dumps(signature,indent=2))
    if not cfg.get('oracle'): rpc('load',cfg)
    mode=' --oracle' if cfg.get('oracle') else ''
    call(prefix+['bash','-c',
       'source /opt/ros/humble/setup.bash && source /work/ws/install/setup.bash && '
       f'python3 /work/src/ros_trials.py --out /work/evaluation/{name} --start {start} --count {count}{mode}'],name)
    episodes=[json.loads((out/f'episode_{seed:05d}.json').read_text()) for seed in range(start,start+count)]
    valid=[r for r in episodes if 'terminal_rmse_rad' in r]
    row={'name':name,**cfg,'n':count,'successes':sum(r['success'] for r in episodes),
         'technical_errors':sum('error' in r for r in episodes),
         'terminal_rmse_mean':mean(r['terminal_rmse_rad'] for r in valid) if valid else None,
         'jerk_rms_mean':mean(r['joint_jerk_rms'] for r in valid) if valid else None,
         'limit_interventions':sum(r.get('command_limit_violations',0) for r in episodes),
         'latency_ms_mean':mean(r['inference_latency_ms_mean'] for r in valid) if valid else None}
    (out/'summary.json').write_text(json.dumps(row,indent=2)); print(json.dumps(row),flush=True)
    return row

def ranking(row):
    # Technical failures always count as failures, and are separately reported.
    return (-row['successes'],row['terminal_rmse_mean'] if row['terminal_rmse_mean'] is not None else float('inf'))

def main():
    call([sys.executable,str(SRC/'act_experiment.py')],'train_all')
    log=(BASE/'policy_service.log').open('w')
    service=subprocess.Popen([sys.executable,str(SRC/'policy_service.py')],stdout=log,stderr=subprocess.STDOUT)
    try:
        for _ in range(60):
            if 'POLICY_SERVICE_READY' in (BASE/'policy_service.log').read_text(): break
            if service.poll() is not None: raise RuntimeError('Policy service failed')
            time.sleep(1)
        oracle=evaluate('val_task_specific_oracle',{'oracle':True},1000,20)
        rows=[]
        for trial in ['lesson_baseline','order_only_200','time_only_200','aligned_200','aligned_2000','diverse_2000']:
            rows.append(evaluate('val_'+trial,{'trial':trial,'n_action_steps':1,'ensemble':None},1000,20))
        winner=min(rows,key=ranking)
        for n,ensemble in [(4,None),(8,None),(1,.01)]:
            cfg={'trial':winner['trial'],'n_action_steps':n,'ensemble':ensemble}
            rows.append(evaluate(f"val_execution_n{n}_ensemble{ensemble}",cfg,1000,20))
        winner=min(rows,key=ranking)
        if winner['successes']==0:
            (BASE/'validation_needs_revision.json').write_text(json.dumps(rows,indent=2))
            print('NO_VALIDATED_SUCCESS: final holdout remains untouched',flush=True)
            return
        selected={k:winner[k] for k in ['trial','n_action_steps','ensemble']}
        selection={'selected':selected,'validation':rows,'test_accessed':False,
                   'checkpoint_sha256':hashlib.sha256((BASE/'results'/winner['trial']/'checkpoint/model.safetensors').read_bytes()).hexdigest()}
        (BASE/'selection_before_test.json').write_text(json.dumps(selection,indent=2))
        print('SELECTION_LOCKED',json.dumps(selected),flush=True)
        test=[]
        for name,cfg in [('test_baseline',{'trial':'lesson_baseline','n_action_steps':1,'ensemble':None}),('test_selected',selected)]:
            test.append(evaluate(name,cfg,2000,50))
        (BASE/'final_results.json').write_text(json.dumps({'validation':rows,'selection':selected,'test':test,'task_specific_oracle_validation':oracle},indent=2))
        print('COMPARISON_COMPLETE',json.dumps(test),flush=True)
    finally:
        service.terminate(); service.wait(timeout=20); log.close()

if __name__=='__main__': main()


In [ ]:
%%writefile /content/pai_ros2/src/analyze_results.py
"""Auditable held-out statistics and plots; no invented or imputed scores."""
import argparse
import json
import math
from pathlib import Path
import numpy as np

def wilson(k,n):
    if not n: raise ValueError('No trials')
    z=1.959963984540054; p=k/n; den=1+z*z/n
    center=(p+z*z/(2*n))/den
    half=z*math.sqrt(p*(1-p)/n+z*z/(4*n*n))/den
    return (0. if k==0 else max(0.,center-half)),(1. if k==n else min(1.,center+half))

def analyze(base):
    result=json.loads((base/'final_results.json').read_text())
    test_start=result.get('test_start_seed',2000)
    a=[json.loads((base/'evaluation/test_baseline'/f'episode_{s:05d}.json').read_text()) for s in range(test_start,test_start+50)]
    b=[json.loads((base/'evaluation/test_selected'/f'episode_{s:05d}.json').read_text()) for s in range(test_start,test_start+50)]
    assert [x['seed'] for x in a]==[x['seed'] for x in b]
    gains=sum(not x['success'] and y['success'] for x,y in zip(a,b))
    regressions=sum(x['success'] and not y['success'] for x,y in zip(a,b))
    n=gains+regressions
    p=min(1.,2*sum(math.comb(n,i) for i in range(min(gains,regressions)+1))/2**n) if n else 1.
    stats={'paired_gains':gains,'paired_regressions':regressions,'mcnemar_exact_two_sided_p':p,
           'scope':'50 paired initial postures from U(-0.08,0.08), fixed task and simulator; no real-robot claim'}
    for name,records in [('baseline',a),('selected',b)]:
        successes=sum(x['success'] for x in records)
        completion=[x['completion_sim_seconds'] for x in records if x['success']]
        stats[name]={'successes':successes,'n':len(records),'wilson95':wilson(successes,len(records)),
                     'technical_errors':sum('error' in x for x in records),
                     'completion_sim_seconds_mean_successes_only':float(np.mean(completion)) if completion else None}
    (base/'statistical_summary.json').write_text(json.dumps(stats,indent=2))
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    fig,axes=plt.subplots(1,2,figsize=(11,4.5),layout='constrained')
    rows=result['validation']; labels=[r['name'].removeprefix('val_') for r in rows]
    axes[0].barh(labels,[r['successes']/r['n'] for r in rows],color='#357ebd')
    axes[0].set(xlim=(0,1.05),xlabel='Success rate (validation)',title='Validation: 20 initial postures')
    for i,r in enumerate(rows): axes[0].text(r['successes']/r['n']+.01,i,f"{r['successes']}/{r['n']}",va='center',fontsize=8)
    scores=[stats[k]['successes']/stats[k]['n'] for k in ['baseline','selected']]
    intervals=np.array([stats[k]['wilson95'] for k in ['baseline','selected']])
    axes[1].bar(['Lesson configuration','Selected ACT'],scores,color=['#7d8894','#138a72'])
    axes[1].errorbar([0,1],scores,yerr=np.maximum(0,np.array([scores-intervals[:,0],intervals[:,1]-scores])),fmt='none',color='black',capsize=5)
    axes[1].set(ylim=(0,1.15),ylabel='Success rate (test)',title='Untouched test: 50 paired initial postures')
    axes[1].text(.5,1.08,'Bars: observed rate; whiskers: 95% Wilson interval',ha='center',fontsize=8)
    fig.savefig(base/'comparison.png',dpi=180); plt.close(fig)
    # Representative seed is predetermined, never selected for appearance.
    fig,axes=plt.subplots(2,1,figsize=(9,6),sharex=True,layout='constrained')
    for records,label in [(a,'Lesson configuration'),(b,'Selected ACT')]:
        row=records[0]
        if not row.get('trace'): continue
        t=np.array([r['t'] for r in row['trace']]); q=np.array([r['q'] for r in row['trace']])
        axes[0].plot(t,q[:,2],label=label)
        err=np.sqrt(np.mean((q-np.array([0,0,.6,0,-.5236]))**2,axis=1))
        axes[1].plot(t,err,label=label)
    axes[0].axhline(.6,color='black',ls='--',lw=1,label='Target')
    axes[0].set(ylabel='Joint 3 (rad)',title=f'Measured Gazebo trace, preselected test seed {test_start}')
    axes[1].set(xlabel='Simulation time (s)',ylabel='Joint RMSE (rad)')
    for ax in axes: ax.legend(); ax.grid(alpha=.2)
    fig.savefig(base/'measured_trace.png',dpi=180); plt.close(fig)
    print(json.dumps(stats,indent=2))
    return stats

if __name__=='__main__':
    p=argparse.ArgumentParser(); p.add_argument('base',type=Path); args=p.parse_args(); analyze(args.base)


In [ ]:
%%writefile /content/pai_ros2/src/summarize_normalization.py
"""Post-training diagnostic, no policy mutation or model selection."""
import json
import numpy as np
from act_experiment import BASE,RecordedDataset,TRIALS

out={}
for name in ['aligned_2000','diverse_2000']:
    dataset=RecordedDataset(BASE/'data/train',TRIALS[name])
    stats=dataset.stats['observation.state']
    out[name]={'state_mean':stats['mean'].tolist(),'state_std':stats['std'].tolist(),
               'uniform_initial_half_width_rad':[.08,.08,.08,.08,0.],
               'normalizer_epsilon':1e-8,
               'normalized_half_width':(np.array([.08,.08,.08,.08,0.])/(stats['std']+1e-8)).tolist()}
(BASE/'normalization_diagnostic.json').write_text(json.dumps(out,indent=2))
print(json.dumps(out,indent=2))


In [ ]:
%%writefile /content/pai_ros2/src/package_artifacts.py
"""Export measured evidence separately from large checkpoints for reliable saving."""
import hashlib
import json
import subprocess
import zipfile
import importlib.metadata
from pathlib import Path

BASE=Path('/content/pai_ros2'); EXPORT=BASE/'exports'; EXPORT.mkdir(exist_ok=True)
metadata={}
for name,path in [('pai_training',BASE/'pai_training'),('lerobot',BASE/'pai_training/lerobot'),('crane_plus',BASE/'ws/src/crane_plus'),('proot',BASE/'proot-src')]:
    metadata[name]=subprocess.check_output(['git','-C',str(path),'rev-parse','HEAD'],text=True).strip()
(BASE/'source_commits.json').write_text(json.dumps(metadata,indent=2))
versions={name:importlib.metadata.version(name) for name in ['torch','torchvision','numpy','datasets','huggingface-hub','draccus','jsonlines','safetensors','einops']}
(BASE/'environment_packages.json').write_text(json.dumps(versions,indent=2))
prefix=json.loads((BASE/'proot_prefix.json').read_text())
ros_versions=subprocess.check_output(prefix+['dpkg-query','-W','-f=${Package} ${Version}\n','ros-humble-*'],text=True)
(BASE/'ros_package_versions.txt').write_text(ros_versions)
selection=json.loads((BASE/'selection_before_test.json').read_text())
(BASE/'selected_policy.json').write_text(json.dumps({
    **selection['selected'],'checkpoint_sha256':selection['checkpoint_sha256'],
    'load_instruction':'Use policy_service.py /load with trial, n_action_steps and ensemble from this file. The training checkpoint config alone does not encode selected execution overrides.'
},indent=2))
files=[]
for pattern in ['*.json','*.png','*.log','*.txt','*.urdf','src/*.py','data/**/*.json','evaluation/**/*.json','results/*/*.json','results/*/*.jsonl']:
    files.extend(BASE.glob(pattern))
files=sorted(set(p for p in files if p.is_file() and p.name!='evidence_manifest.json'))
manifest={str(p.relative_to(BASE)):hashlib.sha256(p.read_bytes()).hexdigest() for p in files}
(BASE/'evidence_manifest.json').write_text(json.dumps(manifest,indent=2))
files.append(BASE/'evidence_manifest.json')
with zipfile.ZipFile(EXPORT/'pai_ros2_act_evidence.zip','w',zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for p in files: z.write(p,p.relative_to(BASE))
selected=selection['selected']['trial']
with zipfile.ZipFile(EXPORT/'pai_ros2_act_checkpoints.zip','w',zipfile.ZIP_DEFLATED,compresslevel=1) as z:
    for trial in sorted({'lesson_baseline',selected}):
        for p in (BASE/'results'/trial/'checkpoint').iterdir():
            if p.is_file(): z.write(p,p.relative_to(BASE))
        p=BASE/'results'/trial/'trial.json'; z.write(p,p.relative_to(BASE))
    z.write(BASE/'selection_before_test.json','selection_before_test.json')
    z.write(BASE/'selected_policy.json','selected_policy.json')
for p in EXPORT.iterdir(): print(p.name,p.stat().st_size,hashlib.sha256(p.read_bytes()).hexdigest(),flush=True)


In [ ]:
import json
protocol={'status': 'registered_before_training', 'date_jst': '2026-11-03', 'lesson_url': 'https://skinny-chord-c58.notion.site/5-ROS-2-2-2-2049a7a1061380fc8928e500eafc57d0', 'task': 'CRANE+ V2 bow: zero arm posture to joint3=0.60 rad, other arm joints=0, gripper=-0.5236', 'scope': 'New experiment separate from already submitted MuJoCo delivery project', 'compute': 'Google Colab T4, existing subscription; no additional purchases', 'source_commits': {'matsuolab/pai_training': 'b4d6b5459066a80433475a83567504988875248a', 'huggingface/lerobot': '768e36660d1408c71118a2760f831c037fbfa17d', 'matsuolab/physicalai_teleop': '0ef2a00fd6d7bb39377969d10f3d26268e447fd9', 'AI-Robot-Book-Humble/crane_plus': '19c1d6069126cd27488bf1bbefd410704bfc515d'}, 'demonstrations': {'provenance': 'Scripted replay of published 0.05-rad keyboard commands, recorded from Gazebo joint feedback; not human demonstrations', 'base_episode_count': 20, 'expanded_episode_count': 80, 'base_seeds': [0, 19], 'expansion_seeds': [20, 79], 'quality_gate': 'finite observations, complete joint names, strictly increasing timestamps, target reached, valid limits; rejected episodes logged'}, 'split_unit': 'whole episode / scene, never random frames', 'validation': {'seeds': [1000, 1019], 'arm_initial_uniform_radians': [-0.08, 0.08]}, 'test': {'seeds': [2000, 2049], 'arm_initial_uniform_radians': [-0.08, 0.08], 'access': 'one final evaluation after model selection; no tuning on test'}, 'success': 'all 5 joint errors <=0.05 rad and max joint speed <=0.10 rad/s continuously for 0.5 simulation seconds, within 8 seconds', 'metrics': ['success_count', 'terminal_joint_rmse_rad', 'completion_sim_seconds', 'executed_joint_jerk_rms', 'command_limit_violations', 'inference_latency_ms'], 'selection': 'validation success first, terminal RMSE second; report latency and smoothness separately, do not hide failed runs', 'planned_trials': [{'name': 'lesson_baseline', 'steps': 200, 'batch_size': 2, 'learning_rate': 0.0001, 'chunk_size': 16, 'n_action_steps': 1, 'action_time_start': -0.1, 'action_time_interval': 0.1, 'training_episodes': 20}, {'name': 'aligned_200', 'steps': 200, 'batch_size': 2, 'action_time_start': 0.0, 'action_time_interval': 0.05, 'training_episodes': 20}, {'name': 'aligned_2000', 'steps': 2000, 'batch_size': 16, 'training_episodes': 20}, {'name': 'diverse_2000', 'steps': 2000, 'batch_size': 16, 'training_episodes': 80}, {'name': 'best_checkpoint_execution', 'n_action_steps_candidates': [1, 4, 8], 'temporal_ensemble_candidates': [None, 0.01]}], 'reproducibility': 'save seed, source hashes, dependency versions, checkpoints, per-episode traces, errors and wall time', 'claim_limit': 'Best among tested configurations under this protocol, never a claim of global optimum. Gazebo reproduction and offline diagnostics are reported separately.', 'stopping_rule': 'Finish planned comparisons and validation selection; add further trials only for a recorded hypothesis. Preserve final holdout; if new development follows test disclosure, create a new untouched test split.', 'pre_training_amendments': [{'time_jst': '2026-11-03T01:38:29.477169+09:00', 'reason': 'Verify feasibility and show the limitation of a single fixed-target task', 'addition': 'Validation-only known-target controller reference, excluded from learned-model selection.'}, {'time_jst': '2026-11-03T01:38:29.477226+09:00', 'reason': 'No human demonstration bags supplied; reproducible simulated collection', 'addition': 'Use actual Gazebo feedback JSON with an in-memory adapter reproducing lesson configuration and conversion conventions, not a byte-identical LeRobot bag conversion; report explicitly.'}, {'time_jst': '2026-11-03T01:39:26.534411+09:00', 'reason': 'Separate joint-order and time-target hypotheses instead of crediting a bundled fix to one cause', 'addition': 'Two 200-step batch-2 ablations: order_only_200 retains original timing and padding; time_only_200 retains original joint order and padding. Both evaluated on validation, eligible for model selection.'}, {'time_jst': '2026-11-03T01:50:05.727834+09:00', 'reason': 'Actual Gazebo probe seed 9999 gives raw order [joint1,joint2,joint4,joint3,hand], matching upstream inference', 'addition': 'Joint-order mismatch hypothesis rejected for this runtime. Retain canonical-order audit trial but never label it a demonstrated bug fix. Adaptive reset must reach <0.005 rad error and low motion for 0.3 seconds, timeout 5 seconds, shared by every method.'}]}
(BASE/"experiment_protocol.json").write_text(json.dumps(protocol,ensure_ascii=False,indent=2))

## 4. 前処理とGPUの動作確認
人工データによるGPUテストは、精度評価とは区別します。

In [ ]:
subprocess.run([sys.executable,"-m","unittest","discover","-s",str(SRC),"-p","test_data_alignment.py"],check=True)
subprocess.run([sys.executable,str(SRC/"smoke_gpu.py")],check=True)

## 5. 実シミュレーションの起動確認とデータ収集
seed 9999は環境確認専用です。学習はseed 0～79を使います。人手の実演ではなく、公開キーボード操作量を参考にしたスクリプトで収集します。

In [ ]:
subprocess.run([sys.executable,str(SRC/"start_colab_runtime.py")],check=True)

In [ ]:
subprocess.run(prefix+["bash","-c","source /opt/ros/humble/setup.bash && source /work/ws/install/setup.bash && python3 /work/src/ros_trials.py --teacher --start 0 --count 80 --out /work/data/train"],check=True)

## 6. 比較学習とモデル選択
学習seedは2026。初回の9設定比較で選んだ diverse_2000 / n_action_steps=1 / ensemble=0.01 を再学習前に固定します。新しい最終評価seed 4000～4049を使い、結果に合わせた調整は行いません。再学習や実行コードを変えた結果を、古い評価結果と混ぜないための検査も入っています。

In [ ]:
%%writefile /content/pai_ros2/src/run_replication.py
"""Rebuild lost temporary artifacts; freeze the first run's chosen configuration.

Uses new final evaluation seeds 4000..4049. No tuning on these results.
The first run's summaries are preserved separately, not mixed with new episodes.
"""
import json,subprocess,sys,time,hashlib
from run_comparison import BASE,SRC,call,evaluate

def main():
    original=json.loads((BASE/'original_run_final_results.json').read_text())
    selected=original['selection']
    assert selected=={'trial':'diverse_2000','n_action_steps':1,'ensemble':.01}
    frozen={'selected':selected,'source':'Initial completed run, saved notebook output',
            'test_start_seed':4000,'test_count':50,'tuning_allowed':False}
    (BASE/'replication_frozen_before_training.json').write_text(json.dumps(frozen,indent=2))
    for name in ['lesson_baseline','diverse_2000']:
        call([sys.executable,str(SRC/'act_experiment.py'),'--trial',name],'train_'+name)
    selection={**frozen,'test_accessed':False,
        'checkpoint_sha256':hashlib.sha256((BASE/'results/diverse_2000/checkpoint/model.safetensors').read_bytes()).hexdigest()}
    (BASE/'selection_before_test.json').write_text(json.dumps(selection,indent=2))
    log=(BASE/'policy_service.log').open('w')
    service=subprocess.Popen([sys.executable,str(SRC/'policy_service.py')],stdout=log,stderr=subprocess.STDOUT)
    try:
        for _ in range(60):
            if 'POLICY_SERVICE_READY' in (BASE/'policy_service.log').read_text():break
            if service.poll() is not None:raise RuntimeError('Policy service failed')
            time.sleep(1)
        oracle=evaluate('val_task_specific_oracle',{'oracle':True},1000,20)
        baseline={'trial':'lesson_baseline','n_action_steps':1,'ensemble':None}
        rows=[evaluate('val_lesson_baseline',baseline,1000,20),evaluate('val_diverse_2000',selected,1000,20)]
        # Configuration was already frozen, independently of this validation.
        test=[evaluate('test_baseline',baseline,4000,50),evaluate('test_selected',selected,4000,50)]
        result={'run_kind':'fixed_configuration_replication','validation':rows,'selection':selected,
                'test':test,'task_specific_oracle_validation':oracle,'test_start_seed':4000,
                'original_run_summary_file':'original_run_final_results.json',
                'incident':'Original run completed 25/50 vs 50/50, but temporary raw files and weights were lost during a native download-save-dialog timeout. Notebook outputs were recovered from Google Drive. This run is a new training and independent evaluation, not recovery of identical original weights.'}
        (BASE/'final_results.json').write_text(json.dumps(result,indent=2))
        print('REPLICATION_COMPLETE',json.dumps(result),flush=True)
    finally:
        service.terminate();service.wait(timeout=20);log.close()

if __name__=='__main__':main()


In [ ]:
original={'validation': [{'name': 'val_lesson_baseline', 'trial': 'lesson_baseline', 'n_action_steps': 1, 'ensemble': None, 'n': 20, 'successes': 12, 'technical_errors': 0, 'terminal_rmse_mean': 0.020532520850612884, 'jerk_rms_mean': 4.269065100030073, 'limit_interventions': 0, 'latency_ms_mean': 13.814827013752023}, {'name': 'val_order_only_200', 'trial': 'order_only_200', 'n_action_steps': 1, 'ensemble': None, 'n': 20, 'successes': 19, 'technical_errors': 0, 'terminal_rmse_mean': 0.021255528952153237, 'jerk_rms_mean': 3.0470416434497563, 'limit_interventions': 0, 'latency_ms_mean': 14.240767836847706}, {'name': 'val_time_only_200', 'trial': 'time_only_200', 'n_action_steps': 1, 'ensemble': None, 'n': 20, 'successes': 13, 'technical_errors': 0, 'terminal_rmse_mean': 0.009779612525750521, 'jerk_rms_mean': 4.281357837526636, 'limit_interventions': 0, 'latency_ms_mean': 15.078091216775084}, {'name': 'val_aligned_200', 'trial': 'aligned_200', 'n_action_steps': 1, 'ensemble': None, 'n': 20, 'successes': 8, 'technical_errors': 0, 'terminal_rmse_mean': 0.024916801343915048, 'jerk_rms_mean': 2.956547790270677, 'limit_interventions': 0, 'latency_ms_mean': 14.694791084436579}, {'name': 'val_aligned_2000', 'trial': 'aligned_2000', 'n_action_steps': 1, 'ensemble': None, 'n': 20, 'successes': 10, 'technical_errors': 0, 'terminal_rmse_mean': 0.12112996785241752, 'jerk_rms_mean': 2.4915932624109383, 'limit_interventions': 0, 'latency_ms_mean': 13.766765608359492}, {'name': 'val_diverse_2000', 'trial': 'diverse_2000', 'n_action_steps': 1, 'ensemble': None, 'n': 20, 'successes': 20, 'technical_errors': 0, 'terminal_rmse_mean': 0.004912693176055122, 'jerk_rms_mean': 2.4386570407901664, 'limit_interventions': 0, 'latency_ms_mean': 13.658260940565047}, {'name': 'val_execution_n4_ensembleNone', 'trial': 'diverse_2000', 'n_action_steps': 4, 'ensemble': None, 'n': 20, 'successes': 20, 'technical_errors': 0, 'terminal_rmse_mean': 0.0047875908781267636, 'jerk_rms_mean': 3.2898265395007424, 'limit_interventions': 0, 'latency_ms_mean': 6.570520588202969}, {'name': 'val_execution_n8_ensembleNone', 'trial': 'diverse_2000', 'n_action_steps': 8, 'ensemble': None, 'n': 20, 'successes': 20, 'technical_errors': 0, 'terminal_rmse_mean': 0.004560912313930988, 'jerk_rms_mean': 3.0051452171243955, 'limit_interventions': 0, 'latency_ms_mean': 5.510499450566331}, {'name': 'val_execution_n1_ensemble0.01', 'trial': 'diverse_2000', 'n_action_steps': 1, 'ensemble': 0.01, 'n': 20, 'successes': 20, 'technical_errors': 0, 'terminal_rmse_mean': 0.004410537477920638, 'jerk_rms_mean': 2.627868539352085, 'limit_interventions': 0, 'latency_ms_mean': 14.142837933323463}], 'selection': {'trial': 'diverse_2000', 'n_action_steps': 1, 'ensemble': 0.01}, 'test': [{'name': 'test_baseline', 'trial': 'lesson_baseline', 'n_action_steps': 1, 'ensemble': None, 'n': 50, 'successes': 25, 'technical_errors': 0, 'terminal_rmse_mean': 0.020929609284336896, 'jerk_rms_mean': 2.7073230080121014, 'limit_interventions': 0, 'latency_ms_mean': 13.459248474417427}, {'name': 'test_selected', 'trial': 'diverse_2000', 'n_action_steps': 1, 'ensemble': 0.01, 'n': 50, 'successes': 50, 'technical_errors': 0, 'terminal_rmse_mean': 0.004578955014129595, 'jerk_rms_mean': 2.482310677571327, 'limit_interventions': 0, 'latency_ms_mean': 13.75600627745584}], 'task_specific_oracle_validation': {'name': 'val_task_specific_oracle', 'oracle': True, 'n': 20, 'successes': 20, 'technical_errors': 0, 'terminal_rmse_mean': 0.007772280162069352, 'jerk_rms_mean': 3.040846626930882, 'limit_interventions': 0, 'latency_ms_mean': 0.0}}
(BASE/"original_run_final_results.json").write_text(json.dumps(original,indent=2))


In [ ]:
subprocess.run([sys.executable,str(SRC/"run_replication.py")],check=True)

## 7. 実測結果の集計と保存
最終評価の成功率には95%区間を付けます。これは固定課題・限定した初期姿勢での結果で、実機や別の動作の汎化性能を示すものではありません。

In [ ]:
assert (BASE/"final_results.json").exists(), "検証段階で改善が必要です。最終評価結果はまだありません"
subprocess.run([sys.executable,str(SRC/"analyze_results.py"),str(BASE)],check=True)
subprocess.run([sys.executable,str(SRC/"summarize_normalization.py")],check=True)
subprocess.run([sys.executable,str(SRC/"package_artifacts.py")],check=True)
print((BASE/"final_results.json").read_text())

In [ ]:
print("成果物: /content/pai_ros2/exports/")
print((BASE/"statistical_summary.json").read_text())
print((BASE/"final_results.json").read_text())
print("右クリック→ダウンロード後、Chromeの保存先ダイアログで保存を完了してください。")

## 参照
- [指定教材](https://skinny-chord-c58.notion.site/5-ROS-2-2-2-2049a7a1061380fc8928e500eafc57d0)
- [教材コード](https://github.com/matsuolab/pai_training)
- [CRANE+ V2](https://github.com/AI-Robot-Book-Humble/crane_plus)
- [LeRobot](https://github.com/huggingface/lerobot)
- [ACT原論文](https://arxiv.org/abs/2304.13705)

上流コードの著作権・ライセンスを維持します。原教材のrosbag変換そのものの完全再現ではなく、実測JSONを使うメモリ上のローダーによる比較です。